# Predicting National CO₂ Emissions Using Economic and Energy Indicators

**Team members:** Dylan Wang, Yipeng Cao, Lani Le  
**Repository:** (https://github.com/yilinw762/group-17)

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [1]:
from pathlib import Path
import json, time, warnings
from datetime import datetime, timezone
from functools import reduce
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
assert (ROOT / 'data').is_dir(), 'Open notebook.ipynb from the repository root directory.'
RAW = ROOT / 'data/raw/world_bank_indicators'
PROCESSED = ROOT / 'data/processed'
FIG = ROOT / 'figures'
for d in (RAW, PROCESSED, FIG): d.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

START_YEAR, END_YEAR = 2000, 2020
INDICATORS = {
    'co2_pc': 'EN.GHG.CO2.PC.CE.AR5',
    'gdp_pc': 'NY.GDP.PCAP.KD',
    'fossil_pct': 'EG.USE.COMM.FO.ZS',
    'renewable_pct': 'EG.FEC.RNEW.ZS',
    'energy_intensity': 'EG.EGY.PRIM.PP.KD',
    'industry_pct': 'NV.IND.TOTL.ZS',
    'urban_pct': 'SP.URB.TOTL.IN.ZS',
}
PREDICTORS = [x for x in INDICATORS if x != 'co2_pc']

## Load data

In [2]:
BASE = 'https://api.worldbank.org/v2'
session = requests.Session()
session.headers.update({'User-Agent': 'DS3001-WDI-course-project/1.0'})

# get all pages of world bank api endpoint, returning a list of records
def get_pages(endpoint, params=None):
    params = {**(params or {}), 'format': 'json', 'per_page': 1000}
    rows = []
    page = 1
    while True:
        for attempt in range(4):
            try:
                r = session.get(f'{BASE}/{endpoint}', params={**params, 'page': page}, timeout=60)
                r.raise_for_status()
                payload = r.json()
                if not isinstance(payload, list) or len(payload) != 2 or not isinstance(payload[0], dict):
                    raise ValueError(f'Bad World Bank response: {str(payload)[:200]}')
                break
            except (requests.RequestException, ValueError):
                if attempt == 3: raise
                time.sleep(2 ** attempt)
        meta, batch = payload
        rows.extend(batch or [])
        if page >= int(meta.get('pages', 1)):
            break
        page += 1
    return rows

# get all countries and their metadata, caching to a csv
def load_countries():
    path = RAW / 'countries.csv'
    if path.exists(): return pd.read_csv(path, dtype={'country_code': str})
    records = get_pages('country')
    rows = [{'country_code': r['id'], 'country': r['name'],
             'region': r['region']['value'], 'income_level': r['incomeLevel']['value']}
            for r in records if r['region']['value'] != 'Aggregates']
    frame = pd.DataFrame(rows)
    frame.to_csv(path, index=False)
    return frame

# get all records for a given indicator, store in csv too
def load_indicator(name, code):
    path = RAW / f'{name}.csv'
    if path.exists():
        return pd.read_csv(path, dtype={'country_code': str})
    records = get_pages(f'country/all/indicator/{code}',
                        {'source': 2, 'date': f'{START_YEAR}:{END_YEAR}'})
    frame = pd.DataFrame([{'country_code': r.get('countryiso3code'),
                           'year': int(r['date']), name: r['value']}
                          for r in records])
    if frame.empty: raise ValueError(f'No records returned for {name}: {code}')
    frame.to_csv(path, index=False)
    return frame

countries = load_countries()
tables = {name: load_indicator(name, code) for name, code in INDICATORS.items()}
manifest = {'provider': 'World Bank World Development Indicators',
            'retrieved_utc': datetime.now(timezone.utc).isoformat(),
            'years': [START_YEAR, END_YEAR], 'indicators': INDICATORS,
            'note': 'Date reflects this run; cached files may have been downloaded earlier.'}
(RAW / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Individual countries/economies in metadata:', len(countries))
for name, df in tables.items(): print(name, len(df), 'nonmissing:', df[name].notna().sum())

Individual countries/economies in metadata: 217
co2_pc 5586 nonmissing: 5271
gdp_pc 5586 nonmissing: 5349
fossil_pct 5586 nonmissing: 4090
renewable_pct 5586 nonmissing: 5427
energy_intensity 5586 nonmissing: 5177
industry_pct 5586 nonmissing: 5116
urban_pct 5586 nonmissing: 5565


## Prepare data

In [ ]:
# Clean data and create features.

## Explore data

In [ ]:
# Summarize and visualize the prepared data.

## Build models

In [ ]:
# Train baseline and candidate models.

## Evaluate models

In [ ]:
# Compare models using the chosen evaluation measures.

## Final results

In [ ]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]